# Local & API-Based Tool Calling

_Experimenting with local tools and external APIs that provides additional capabilities that an agent can perform to achieve a desired outcomes._

Based on the end user's query, the agent runs mathimatical operations as local tools or external APIs to provide an agent access to the most recent information from Wikipedia.

In [1]:
# Imports packages

from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_ollama.chat_models import ChatOllama
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun

import time

In [2]:
# Sets endpoints for Ollama models to be available over web requests.

OLLAMA_MODEL = "llama3.2:3b"
OLLAMA_ENDPOINT = "http://localhost:11434/"

## Tools


**Mathematical Functions as Local Tools**

Uses decorator to convert Python functions to LangChain tools that the agent will be using.

In [3]:
@tool
def add(x: float, y: float) -> float:
    """Adds 'x' and 'y'."""
    return f"The sum of {x} + {y} is {x + y}."

@tool
def subtract(x: float, y: float) -> float:
    """Subtracts 'y' from 'x'."""
    return f"{x} minus {y} equals {x - y}."

@tool
def multiply(x: float, y: float) -> float:
    """Multiplies 'x' with 'y'."""
    return f"{x} times {y} equals {x * y}."

**Wikipedia API as External Tool to Retrieve Information from Wikipedia**

In [4]:
wikipedia = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=4000)   # Requires wikipedia package
    )

In [5]:
tools = {
    "add": add,
    "subtract": subtract,
    "multiply": multiply,
    "wikipedia": wikipedia
}

## Agent

In [6]:
# Instruction to be used to set agent's behaviour

agent_instructions = """You are a simple assistant who can only help with following 
math services and Wikipedia:

1) Math services
    a) add
    b) subtract
    c) multiply

2) Wikipedia

Rules:
- If a request is outside the above mentioned services and search, politely decline, 
inform user what services and searh you offer and let her re-enter the query.
- The query, especially for math services, should be complete in itself meaning
the query inself should representative of mathematical operation that will be performed
and parameters the operation to perform on.
- Do not call tools until all required fields are explicitly provided by the user.
- For every new request, fetch parameter values from the user again. Do not reuse values from prior turns.

For math operations:
- Required field: x, y
- If an parameter is missing, gets input from the user
- If all parameters are present, call add(x, y), subtract(x, y) or multiply(x, y), as applicable.

For Wikipedia:
- Required fields: query
- If user query is present, call Wikipedia tool with query parameter directly.

For tools:
- For tool calls, fill 'tool_calls' structure instead of JSON output in your response.

For response
- Prapare your response from tools messages, if exist.
"""

In [7]:
# Initializes chat client
client = ChatOllama(model=OLLAMA_MODEL, 
                    reasoning=False,
                    base_url=OLLAMA_ENDPOINT
                    )

# Binds tools to the chat client
client_with_tools = client.bind_tools([add, subtract, multiply, wikipedia])

In [8]:
# Maintains a list of all messages that flow across interactions
# and initializing it with intruction (as a SystemMessage) that is specific to agent.

messages = [
    SystemMessage(content=agent_instructions)
    ]

In [9]:
# User may provide one or multiple queries in single turn to be sent to client

query = "What is 150+50? And 30*40? What is the capital of India?"

messages.append(HumanMessage(query))                # Appends user query (as a HumanMessage) into message list

In [10]:
# Passes messages (as a single input) to model and receives response [THIS MAY TAKE SEVERAL SECONDS]
client_message = client_with_tools.invoke(messages)

In [11]:
# Displays the (structured) model-returned message to analyze element `tool_calls` to check
# if model has returned the correct tool name(s) with argument(s) for client to call those later
display(client_message)

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-06-05T08:03:53.294500901Z', 'done': True, 'done_reason': 'stop', 'total_duration': 14158735064, 'load_duration': 103905422, 'prompt_eval_count': 598, 'prompt_eval_duration': 8923639786, 'eval_count': 61, 'eval_duration': 5074925497, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--019e96cf-27fe-74a3-bd96-b3a0a16db4d9-0', tool_calls=[{'name': 'add', 'args': {'x': '150', 'y': '50'}, 'id': '23cb28b1-e609-4553-92a6-c8c7c58d2cab', 'type': 'tool_call'}, {'name': 'multiply', 'args': {'x': '30', 'y': '40'}, 'id': '4967db42-e5db-47c6-a556-3a0e5c30afe2', 'type': 'tool_call'}, {'name': 'wikipedia', 'args': {'query': 'capital of India'}, 'id': 'e497edd0-6b96-4ad4-b36c-7eddfec2b977', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 598, 'output_tokens': 61, 'total_tokens': 659})

In [12]:
messages.append(client_message)     # Appends last received message into message list

In [ ]:
# Now, client loops over each tool name to call the tool manually for execution

for tool_call in client_message.tool_calls:
    print(f"Tool call (detailed): {tool_call}")
    tool_name = tool_call["name"].lower()               # Extracts tool name and converts to lower case
    selected_tool = tools[tool_name]                    # Gets pointer to tool
    
    """
    NOTE: Conditional tool execution was not required, but it was done due to intermitent
    failure observed in Wikipedia tool while its exection over argument with full tool details.
    For example: 
        wikipedia.invoke({'query': 'capital of India'}) works, but
        wikipedia.invoke({'name': 'wikipedia', 'args': {'query': 'capital of India'}, 'id': 'fb82e79c-fd76-4384-ab0d-3a1a99bc4626', 'type': 'tool_call'}) doesn't
   
    """
    
    if tool_name == wikipedia:
        response = selected_tool.invoke(tool_call.get("args"))
        tool_message = ToolMessage(response)
        print(f"Tool name: '{tool_call["name"]}', \nArguments: {tool_call["args"]}, \nTool Output: {tool_message}\n")
    else:
        tool_message = selected_tool.invoke(tool_call)
        print(f"Tool: '{tool_call["name"]}', \nArguments: {tool_call["args"]}, \nTool Output: {tool_message.content}\n")
    
    messages.append(tool_message)                           # Message received from tool gets added to message list

Tool calle (detailed): {'name': 'add', 'args': {'x': '150', 'y': '50'}, 'id': '23cb28b1-e609-4553-92a6-c8c7c58d2cab', 'type': 'tool_call'}
Tool: 'add', 
Arguments: {'x': '150', 'y': '50'}, 
Tool Output: The sum of 150.0 + 50.0 is 200.0.

Tool calle (detailed): {'name': 'multiply', 'args': {'x': '30', 'y': '40'}, 'id': '4967db42-e5db-47c6-a556-3a0e5c30afe2', 'type': 'tool_call'}
Tool: 'multiply', 
Arguments: {'x': '30', 'y': '40'}, 
Tool Output: 30.0 times 40.0 equals 1200.0.

Tool calle (detailed): {'name': 'wikipedia', 'args': {'query': 'capital of India'}, 'id': 'e497edd0-6b96-4ad4-b36c-7eddfec2b977', 'type': 'tool_call'}
Tool: 'wikipedia', 
Arguments: {'query': 'capital of India'}, 
Tool Output: Page: National Capital Region (India)
Summary: The National Capital Region (NCR; Rāṣṭrīya Rājadhānī Kṣetra) is a region centred on the city of Delhi, a special union territory of India that hosts the country's capital city New Delhi. It encompasses the entirety of Delhi and a number of adjac

In [ ]:
final_response = client_with_tools.invoke(messages)         # Passes all messages back to model to generate final response
print(f"Final Agent Response: {final_response.content}")    # Prints final response that the user would see

Final Agent Response: Here are the results of the calculations and the answer to your question about the capital of India:

- The sum of 150 + 50 is 200.
- The product of 30 * 40 is 1200.
- The capital of India is New Delhi, which is part of the National Capital Region (NCR).
